# Train YOLO26s on VEYAKSH traffic

Select a GPU runtime in Colab. For Drive storage, run all cells and approve the Drive prompt. For a session without Drive, upload `yolo26_training_inputs.zip` through the **Files** sidebar to `/content`, wait for the upload to finish, then run all cells. Both modes use the same Task 2.6 training and validation splits.

Drive mode saves checkpoints and evaluation outputs directly to Drive. Upload mode saves them under `/content/yolo26_results` and downloads a results ZIP in the final cell. Runtime files expire when the VM is deleted. To resume an interrupted run, set `RESUME_FROM` below to a saved `weights/last.pt`.

In [ ]:
%pip -q install "ultralytics==8.4.197" "onnx>=1.12,<2" onnxruntime

In [ ]:
import csv
import json
import shutil
import tempfile
from pathlib import Path

import torch
import yaml
from google.colab import drive, files
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → GPU and reconnect.")
BUNDLE = Path("/content/yolo26_training_inputs.zip")
using_upload = BUNDLE.is_file()
if using_upload:
    inputs = Path(tempfile.mkdtemp(prefix="veyaksh_inputs_"))
    shutil.unpack_archive(BUNDLE, inputs)
    SOURCE = inputs / "traffic_dataset"
    BASELINE = inputs / "models/best.pt"
    INITIAL = inputs / "models/yolo26s.pt"
    PROJECT = BUNDLE.parent / "yolo26_results"
else:
    print("Approve Drive access, or upload yolo26_training_inputs.zip to /content and rerun this cell.", flush=True)
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive")
    SOURCE = DRIVE / "TrafficDetection/results/task2_6_final_yolo11s_v1/traffic_dataset"
    BASELINE = DRIVE / "VEYAKSH_TRAFFIC_DATASET/models/best.pt"
    INITIAL = "yolo26s.pt"
    PROJECT = DRIVE / "TrafficDetection/results/yolo26s"
RESUME_FROM = None  # Or Path("/content/drive/MyDrive/.../weights/last.pt")
DEVICE = 0
print(torch.cuda.get_device_name(DEVICE), "| Results directory:", PROJECT)

## Check the original split

Copy the dataset to the GPU runtime for faster reads and rewrite its local root. The checks below use the seven class IDs and the frame and annotation counts recorded for Task 2.6.

In [ ]:
NAMES = dict(enumerate(["two_wheeler", "auto", "e_rickshaw", "car", "bus_truck", "cart", "pedestrian"]))
HOLDOUT = "muzaffarnagar_clip03"
if not BASELINE.is_file():
    raise FileNotFoundError(BASELINE)
if YOLO(str(BASELINE)).names != NAMES:
    raise ValueError("Check the class IDs in the YOLO11s baseline.")
source_cfg = yaml.safe_load((SOURCE / "data.yaml").read_text(encoding="utf-8-sig"))
names = source_cfg["names"]
names = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
if names != NAMES:
    raise ValueError(f"Unexpected dataset class map {names}")

local = SOURCE if using_upload else Path(tempfile.mkdtemp(prefix="veyaksh_traffic_"))
for split, expected in (("train", 2079), ("val", 61)):
    images = local / "images" / split
    labels = local / "labels" / split
    if not using_upload:
        shutil.copytree(SOURCE / "images" / split, images)
        shutil.copytree(SOURCE / "labels" / split, labels)
    stems = [p.stem for p in images.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    if len(stems) != expected or set(stems) != {p.stem for p in labels.glob("*.txt")}:
        raise ValueError(f"Expected {expected} labeled images in {split}")
    clip_frames = sum(stem.startswith(f"shardul_{HOLDOUT}_") for stem in stems)
    if clip_frames != (61 if split == "val" else 0):
        raise ValueError(f"Unexpected {HOLDOUT} frame count in {split}: {clip_frames}")
val_boxes = sum(len(p.read_text().splitlines()) for p in (local / "labels/val").glob("*.txt"))
if val_boxes != 12152:
    raise ValueError(f"Unexpected holdout annotation count {val_boxes}")
DATA = local / "data.yaml"
DATA.write_text(yaml.safe_dump({"path": str(local), "train": "images/train", "val": "images/val", "names": NAMES}))
print(f"Verified 2079 training images and 61 validation images with {val_boxes} boxes.")

In [ ]:
if RESUME_FROM is None:
    candidate = YOLO(str(INITIAL))
    candidate.train(data=str(DATA), device=DEVICE, epochs=50, patience=10, imgsz=640,
                    batch=16, workers=2, seed=0, project=str(PROJECT), name="train")
else:
    # The local data directory changes across Colab sessions.
    candidate = YOLO(str(RESUME_FROM))
    candidate.train(resume=True, data=str(DATA), device=DEVICE)

BEST = Path(candidate.trainer.best)
RUN = BEST.parents[1]
print("Best seven-class YOLO26s checkpoint:", BEST)

## Compare both trained checkpoints on the same frames

Use the same device, input size, batch size, precision and detection limit for both models. The CSV records accuracy per class and validation-stage times per image. These times exclude video decoding, tracking and event processing.

In [ ]:
import ultralytics

metrics = []
timings = []
for label, weights in (("YOLO11s", BASELINE), ("YOLO26s", BEST)):
    model = YOLO(str(weights))
    if model.names != NAMES:
        raise ValueError(f"Unexpected class IDs in {weights}")
    result = model.val(data=str(DATA), split="val", device=DEVICE, imgsz=640, batch=1,
                       conf=0.001, iou=0.7, max_det=300, quantize=None, workers=0,
                       project=str(RUN), name=f"val_{label}", plots=False)
    box = result.box
    def record(class_name, precision, recall, ap50, ap):
        metrics.append(dict(model=label, class_name=class_name, precision=float(precision),
                            recall=float(recall), mAP50=float(ap50), mAP50_95=float(ap)))
    record("all", box.mp, box.mr, box.map50, box.map)
    for index, class_id in enumerate(box.ap_class_index):
        record(NAMES[int(class_id)], box.p[index], box.r[index], box.ap50[index], box.ap[index])
    timings.append(dict(model=label, weights=str(weights), backend="PyTorch FP32",
                        device=torch.cuda.get_device_name(DEVICE) if DEVICE != "cpu" else "cpu",
                        ultralytics=ultralytics.__version__, imgsz=640, batch=1,
                        preprocess_ms=result.speed["preprocess"], inference_ms=result.speed["inference"],
                        postprocess_ms=result.speed["postprocess"]))

for filename, rows in (("heldout_metrics.csv", metrics), ("latency_ms_per_image.csv", timings)):
    with (RUN / filename).open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
print(json.dumps([r for r in metrics if r["class_name"] == "all"] + timings, indent=2))

## Export the edge candidate

The ONNX export uses the YOLO26 end-to-end detection head and embeds the seven class names. The PyTorch validation results above apply to the PyTorch checkpoints. Compare the exported ONNX on the deployment laptop using the same clip and tracker settings as the current ONNX model.

In [ ]:
onnx_file = Path(YOLO(str(BEST)).export(format="onnx", imgsz=640, batch=1, dynamic=False,
                                        opset=17, simplify=False, nms=False, device="cpu"))
print("ONNX candidate:", onnx_file)
print("Metrics and validation timing:", RUN)

edge = YOLO(str(onnx_file), task="detect")
edge_val = edge.val(data=str(DATA), split="val", device="cpu", imgsz=640, batch=1,
                    conf=0.001, iou=0.7, max_det=300, workers=0, plots=False,
                    project=str(RUN), name="val_yolo26s_onnx")
edge_scores = {"weights": str(onnx_file), "heldout_clip": HOLDOUT,
               "mAP50": float(edge_val.box.map50), "mAP50_95": float(edge_val.box.map),
               "AP50_95_by_class": {NAMES[int(i)]: float(ap) for i, ap in
                                    zip(edge_val.box.ap_class_index, edge_val.box.ap)}}
(RUN / "yolo26s_onnx_metrics.json").write_text(json.dumps(edge_scores, indent=2))
print(json.dumps(edge_scores, indent=2))

In [ ]:
if using_upload:
    archive = shutil.make_archive(str(RUN.parent / f"yolo26s_{RUN.name}_results"), "zip", RUN.parent, RUN.name)
    print("Download your results before the session ends:", archive)
    files.download(archive)